# distributed-sampler-shard — ex2: set_epoch reshuffles AND preserves disjoint+coverage

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `distributed-sampler-shard`. Running the final beacon cell reports progress against the `Distributed: DistributedSampler shard` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: DistributedSampler shard` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`distributed-sampler-shard`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "distributed-sampler-shard"
DD_SUBTOPIC = "Distributed: DistributedSampler shard"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `set_epoch` — the across-epochs reshuffle contract

Ex1 collected the shard indices for epoch 0. The deepening: **does `set_epoch(e)` actually produce a DIFFERENT permutation per epoch**, and do the cross-epoch shards still satisfy the disjoint + coverage invariants?

```python
sampler.set_epoch(0); shards_e0 = list(sampler)
sampler.set_epoch(1); shards_e1 = list(sampler)
# shards_e0 != shards_e1   (different permutation, with high probability)
# but per-epoch invariants still hold per rank
```

**Why per-epoch determinism still matters.** Within ONE epoch, every rank's sampler uses the SAME (`epoch` + `seed`) → SAME base permutation. Each rank then slices its strided shard. The disjoint property is a consequence of `index % num_replicas == rank` — true for any permutation.

**Coverage with padding.** When `len(dataset) % world_size != 0`, the sampler pads the permutation by wrapping from the start. So the union of all per-rank shards has length `ceil(N / W) * W`, with the first `(ceil(N/W) * W - N)` indices appearing twice. Coverage is exact only when `N` divides evenly.

**Forgetting `set_epoch` is silent.** Training appears to work — every rank still sees a stable disjoint shard. But that shard is the same every epoch. The model sees the same batch order forever and convergence is wrong.

### Exercise 2 — set_epoch reshuffles AND preserves disjoint+coverage

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze `DistributedSampler` across multiple epochs by calling `set_epoch(e)` before each iteration, returning a nested `list[epoch][rank]` of indices so the test can verify that permutations differ across epochs while each epoch still satisfies the disjoint + coverage invariants.
> Keywords: distributed-sampler, set_epoch, reshuffle, disjoint, coverage
> ```

**KCs targeted:** `set-epoch-changes-permutation`, `per-epoch-disjoint-and-coverage-hold`

Implement `ex2_collect_shards_multi_epoch(dataset, world_size, seed, num_epochs)`. Multi-epoch sharding inspector:

1. Outer loop: `for epoch in range(num_epochs)`.
2. Inner loop: `for rank in range(world_size)`. For each `(epoch, rank)`:
   a. Build `DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True, seed=seed)`.
   b. Call `sampler.set_epoch(epoch)` BEFORE iterating.
   c. Collect indices: `list(sampler)`.
3. Return a `list[list[list[int]]]` of shape `[num_epochs][world_size]`.

Inputs:
- `dataset`: any Dataset (uses only `len(dataset)`).
- `world_size`: int >= 1.
- `seed`: int.
- `num_epochs`: int >= 1.

Output: nested list — `result[epoch][rank]` is the index list for rank `rank` on epoch `epoch`.

**This drill does not call `dist.*` at all.** `DistributedSampler` is a pure-iterator construct that takes `(rank, num_replicas)` as args — no process group needed. The challenge is `set_epoch` discipline + per-epoch verification.

In [ ]:
def ex2_collect_shards_multi_epoch(dataset, world_size: int, seed: int, num_epochs: int) -> list:
    from torch.utils.data.distributed import DistributedSampler
    out = []
    for epoch in range(num_epochs):
        epoch_shards = []
        for rank in range(world_size):
            sampler = DistributedSampler(
                dataset,
                num_replicas=world_size,
                rank=rank,
                shuffle=True,
                seed=seed,
            )
            sampler.set_epoch(epoch)
            epoch_shards.append(list(sampler))
        out.append(epoch_shards)
    return out


<details><summary>Solution</summary>

```python
def ex2_collect_shards_multi_epoch(dataset, world_size: int, seed: int, num_epochs: int) -> list:
    from torch.utils.data.distributed import DistributedSampler
    out = []
    for epoch in range(num_epochs):
        epoch_shards = []
        for rank in range(world_size):
            sampler = DistributedSampler(
                dataset,
                num_replicas=world_size,
                rank=rank,
                shuffle=True,
                seed=seed,
            )
            sampler.set_epoch(epoch)
            epoch_shards.append(list(sampler))
        out.append(epoch_shards)
    return out
```

**`set_epoch` BEFORE iteration, not after.** The shuffle RNG is seeded at the start of `__iter__`, using `(self.epoch, self.seed)` as inputs. Calling `set_epoch` after consuming the iterator does nothing — next iteration's RNG will use the new value, but the iterator you just collected is already locked in.

**Reconstructing the base permutation by interleaving.** `DistributedSampler` builds `perm = torch.randperm(N, generator=g)` (or `perm + padding`) and slices it as `perm[rank::num_replicas]`. Interleaving the per-rank shards by position recovers the full perm — a useful debugging trick.

**Padding behavior is the default.** When `len(dataset) % world_size != 0`, the sampler pads via `perm += perm[: total - N]` (wrap from the start) so every rank gets exactly `ceil(N/W)` items. To drop the tail instead: `DistributedSampler(..., drop_last=True)`. We don't drill drop_last here — it's a separate atom.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()